# 🏸 Badminton Action Classification — CNN-BiLSTM + K-Fold Ensemble
### Target: 94-95% | Dataset: shenhuichang/badminton-storke-video

> ⚡ `Runtime → Change runtime type → T4 GPU`  →  `Runtime → Run All`

**Stack**: YOLOv8-Pose · Rich Features (pos+vel+acc) · CNN-BiLSTM-Attention · 5-Fold Ensemble · TTA

| Stage | Gain |
|---|---|
| Baseline BiLSTM | 63% |
| + Augmentation | 74% |
| + Rich features (vel+acc) | +5% |
| + CNN-BiLSTM-Attention | +5% |
| + 5-Fold Ensemble + TTA | +5-8% |
| **Target** | **94-95%** |

## Step 1 — GPU + Install

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {torch.cuda.get_device_name(0) if device=="cuda" else "CPU — switch to T4 GPU!"}')

!pip install -q ultralytics kagglehub scikit-learn seaborn tqdm
print('✅ Packages ready')

## Step 2 — Kaggle Download

In [ ]:
import os

# ══════════════════════════════════════════════════════
os.environ['KAGGLE_USERNAME'] = 'YOUR_USERNAME'   # ← replace
os.environ['KAGGLE_KEY']      = 'YOUR_API_KEY'    # ← replace
# ══════════════════════════════════════════════════════

import kagglehub
path = kagglehub.dataset_download('shenhuichang/badminton-storke-video')
print('Dataset path:', path)
for root,_,files in os.walk(path):
    vids=[f for f in files if f.lower().endswith(('.mp4','.avi','.mov','.mkv'))]
    if vids: print(f'  📁 {os.path.basename(root)}/  ({len(vids)} videos)')

## Step 3 — Global Config

In [ ]:
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_ROOT   = path
RAW_KP_DIR     = '/content/kp_raw'      # original (10,34)
RICH_KP_DIR    = '/content/kp_rich'     # enriched (10,102)
AUG_RICH_DIR   = '/content/kp_aug'      # augmented (10,102)
MODEL_DIR      = '/content/models'
RESULTS_DIR    = '/content/results'
for d in [RAW_KP_DIR,RICH_KP_DIR,AUG_RICH_DIR,MODEL_DIR,RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Classes ───────────────────────────────────────────────────────────────────
ACTION_CLASSES = [
    'forehand_drive','forehand_lift','forehand_net_shot',
    'forehand_clear','backhand_drive','backhand_net_shot',
]
NUM_CLASSES = len(ACTION_CLASSES)

# ── Features ──────────────────────────────────────────────────────────────────
FRAMES_PER_VIDEO = 15      # ↑ from 10 → richer temporal info
NUM_KP           = 17      # YOLOv8 COCO
RAW_SIZE         = NUM_KP * 2      # 34
INPUT_SIZE       = RAW_SIZE * 3    # 102 = pos + vel + acc

# ── Augmentation ─────────────────────────────────────────────────────────────
AUGMENT_FACTOR = 50

# ── CNN-BiLSTM-Attention ──────────────────────────────────────────────────────
CNN_CHANNELS  = [128, 256]
HIDDEN_SIZE   = 256
NUM_LAYERS    = 2
MHA_HEADS     = 8
DROPOUT       = 0.4
FC_HIDDEN     = 128

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE     = 64
NUM_EPOCHS     = 200
LEARNING_RATE  = 1e-3
WEIGHT_DECAY   = 5e-4
EARLY_STOP_PAT = 35
N_FOLDS        = 5
RANDOM_SEED    = 42
VIDEO_EXTS     = {'.mp4','.avi','.mov','.mkv','.wmv'}

print(f'✅ Config | device: {device}')
print(f'   Frames/video : {FRAMES_PER_VIDEO}')
print(f'   Input size   : {INPUT_SIZE}  (pos + vel + acc)')
print(f'   Augment      : {AUGMENT_FACTOR}×  →  ~{836*AUGMENT_FACTOR:,} samples')
print(f'   K-Folds      : {N_FOLDS}')

## Step 4 — Extract Raw Keypoints (YOLOv8)

In [ ]:
NAME_MAP = {}
for c in ACTION_CLASSES:
    for v in [c,c.replace('_',' '),c.replace('_','-'),c.replace('_',''),
               c.title(),c.replace('_',' ').title()]:
        NAME_MAP[v.lower()] = c

class_to_folder = {}
for dirpath,_,filenames in os.walk(DATASET_ROOT):
    fname = os.path.basename(dirpath).lower().replace('-','_').replace(' ','_')
    canon = NAME_MAP.get(fname) or NAME_MAP.get(fname.replace('_',''))
    if canon and canon not in class_to_folder:
        vids = [f for f in filenames if os.path.splitext(f)[1].lower() in VIDEO_EXTS]
        if vids: class_to_folder[canon] = dirpath

for cls in ACTION_CLASSES:
    f = class_to_folder.get(cls)
    n = len([x for x in os.listdir(f) if os.path.splitext(x)[1].lower() in VIDEO_EXTS]) if f else 0
    print(f'  {"✅" if f else "❌"} {cls:<25} {n} videos')

In [ ]:
from ultralytics import YOLO
import cv2
from tqdm.notebook import tqdm

pose_model = YOLO('yolov8n-pose.pt')
print(f'✅ YOLOv8n-pose | {FRAMES_PER_VIDEO} frames/video | {RAW_SIZE} features/frame')

def extract_video(video_path):
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0: cap.release(); return None
    W = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    H = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    frames = []
    for idx in np.linspace(0, total-1, FRAMES_PER_VIDEO, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret: frames.append(frame)
    cap.release()
    if not frames: return None
    seq = np.zeros((FRAMES_PER_VIDEO, RAW_SIZE), dtype=np.float32)
    for i, frame in enumerate(frames):
        res = pose_model(frame, verbose=False)
        if res and res[0].keypoints is not None:
            kps = res[0].keypoints.xy
            if kps.shape[0] > 0:
                pts = kps[0].cpu().numpy()
                pts[:,0] /= W; pts[:,1] /= H
                seq[i] = pts.flatten()
    return seq

saved = skipped = failed = 0
for cls in ACTION_CLASSES:
    src = class_to_folder.get(cls)
    if src is None: continue
    dst = os.path.join(RAW_KP_DIR, cls); os.makedirs(dst, exist_ok=True)
    vids = [f for f in os.listdir(src) if os.path.splitext(f)[1].lower() in VIDEO_EXTS]
    print(f'[{cls}] {len(vids)} videos')
    for vf in tqdm(vids, desc=cls, leave=False):
        out = os.path.join(dst, os.path.splitext(vf)[0]+'.npy')
        if os.path.exists(out): skipped+=1; continue
        kp = extract_video(os.path.join(src, vf))
        if kp is not None: np.save(out, kp); saved+=1
        else: failed+=1

total = sum(len(os.listdir(os.path.join(RAW_KP_DIR,c)))
            for c in ACTION_CLASSES if os.path.isdir(os.path.join(RAW_KP_DIR,c)))
print(f'\n✅ Raw keypoints: {total} files | shape=(15,{RAW_SIZE}) | saved:{saved} skipped:{skipped}')

## Step 5 — Rich Features + 50× Augmentation

In [ ]:
def enrich(seq):
    """(T,34) → (T,102): concatenate position + velocity + acceleration."""
    vel  = np.zeros_like(seq); vel[1:]  = seq[1:] - seq[:-1]
    acc  = np.zeros_like(seq); acc[1:]  = vel[1:] - vel[:-1]
    return np.concatenate([seq, vel, acc], axis=1).astype(np.float32)


def augment_pos(pos, aug_id):
    """Apply one of 25 augmentations to raw (T,34) position array."""
    pos = pos.copy(); T, F = pos.shape; aid = aug_id % 25
    if   aid== 0: pos += np.random.normal(0,0.005,pos.shape)
    elif aid== 1: pos += np.random.normal(0,0.015,pos.shape)
    elif aid== 2: pos += np.random.normal(0,0.030,pos.shape)
    elif aid== 3:
        for j in range(0,F,2):
            std=np.random.uniform(0.002,0.025)
            pos[:,j]+=np.random.normal(0,std,T); pos[:,j+1]+=np.random.normal(0,std,T)
    elif aid== 4: pos[:,0::2]=1.0-pos[:,0::2]
    elif aid== 5: pos[:,0::2]=1.0-pos[:,0::2]; pos+=np.random.normal(0,0.010,pos.shape)
    elif aid== 6: pos*=np.random.uniform(0.80,1.20)
    elif aid== 7: pos[:,0::2]*=np.random.uniform(0.85,1.15); pos[:,1::2]*=np.random.uniform(0.85,1.15)
    elif aid== 8: pos[:,0::2]+=np.random.uniform(-0.08,0.08); pos[:,1::2]+=np.random.uniform(-0.08,0.08)
    elif aid== 9: pos*=np.random.uniform(0.88,1.12); pos[:,0::2]+=np.random.uniform(-0.05,0.05)
    elif aid==10:
        a=np.random.uniform(-20,20)*np.pi/180; ca,sa=np.cos(a),np.sin(a)
        cx,cy=pos[:,0::2].mean(),pos[:,1::2].mean()
        x,y=pos[:,0::2]-cx,pos[:,1::2]-cy
        pos[:,0::2]=ca*x-sa*y+cx; pos[:,1::2]=sa*x+ca*y+cy
    elif aid==11:
        a=np.random.uniform(-35,35)*np.pi/180; ca,sa=np.cos(a),np.sin(a)
        cx,cy=pos[:,0::2].mean(),pos[:,1::2].mean()
        x,y=pos[:,0::2]-cx,pos[:,1::2]-cy
        pos[:,0::2]=ca*x-sa*y+cx; pos[:,1::2]=sa*x+ca*y+cy
    elif aid==12: pos=pos[::-1].copy()
    elif aid==13:
        idx=np.linspace(0,T-1,T//2,dtype=int); slow=pos[idx]
        pos=np.vstack([slow,slow[:T-len(slow)]])
    elif aid==14:
        fast=pos[::2]; pad=np.tile(pos[-1:],(T-len(fast),1)); pos=np.vstack([fast,pad])[:T]
    elif aid==15: pos=pos[np.sort(np.random.choice(T,T,replace=True))]
    elif aid==16: s=np.random.randint(1,4); pos=np.vstack([pos[s:],np.zeros((s,F),np.float32)])
    elif aid==17:
        k=np.array([0.25,0.5,0.25])
        for f in range(F): pos[:,f]=np.convolve(pos[:,f],k,mode='same')
    elif aid==18: pos[np.random.choice(T,np.random.randint(1,4),replace=False)]=0.0
    elif aid==19:
        for j in np.random.choice(F//2,np.random.randint(2,6),replace=False):
            pos[:,j*2]=pos[:,j*2+1]=0.0
    elif aid==20:
        pos[:,0::2]=1.0-pos[:,0::2]; pos*=np.random.uniform(0.9,1.1)
        pos+=np.random.normal(0,0.008,pos.shape)
    elif aid==21: pos=pos[::-1].copy(); pos*=np.random.uniform(0.9,1.1); pos+=np.random.normal(0,0.008,pos.shape)
    elif aid==22:
        pos[:,0::2]+=np.random.uniform(-0.05,0.05)
        s=np.random.randint(1,3); pos=np.vstack([pos[s:],np.zeros((s,F),np.float32)])
        pos+=np.random.normal(0,0.008,pos.shape)
    elif aid==23:
        pos[:,0::2]=1.0-pos[:,0::2]
        idx=np.linspace(0,T-1,T//2,dtype=int); slow=pos[idx]
        pos=np.vstack([slow,slow[:T-len(slow)]]); pos+=np.random.normal(0,0.010,pos.shape)
    elif aid==24:
        pos+=np.random.normal(0,0.008,pos.shape)
        if np.random.rand()<0.5: pos[:,0::2]=1.0-pos[:,0::2]
        pos*=np.random.uniform(0.90,1.10); pos[:,0::2]+=np.random.uniform(-0.04,0.04)
    return np.clip(pos,0.0,1.0).astype(np.float32)


def mixup(a, b, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    return np.clip(lam*a+(1-lam)*b, 0, 1).astype(np.float32)


# ── Build augmented rich dataset ──────────────────────────────────────
total_aug = 0
for cls in ACTION_CLASSES:
    src = os.path.join(RAW_KP_DIR, cls)
    dst = os.path.join(AUG_RICH_DIR, cls)
    os.makedirs(dst, exist_ok=True)
    if not os.path.isdir(src): continue
    files = sorted([f for f in os.listdir(src) if f.endswith('.npy')])
    seqs  = [np.load(os.path.join(src,f)).astype(np.float32) for f in files]
    print(f'[{cls}] {len(files)} → {len(files)*(AUGMENT_FACTOR+3)}')
    for fname, seq in tqdm(zip(files,seqs), total=len(files), desc=cls, leave=False):
        stem = os.path.splitext(fname)[0]
        np.save(os.path.join(dst,f'{stem}_orig.npy'), enrich(seq)); total_aug+=1
        for k in range(AUGMENT_FACTOR-1):
            np.save(os.path.join(dst,f'{stem}_a{k:02d}.npy'), enrich(augment_pos(seq,k))); total_aug+=1
        for m in range(3):
            j=np.random.randint(len(seqs))
            np.save(os.path.join(dst,f'{stem}_mx{m}.npy'), enrich(mixup(seq,seqs[j]))); total_aug+=1

print(f'\n✅ Augmented dataset: {total_aug:,} samples  (was 836)')
print(f'   Feature shape: ({FRAMES_PER_VIDEO},{INPUT_SIZE}) = pos+vel+acc')
for cls in ACTION_CLASSES:
    d=os.path.join(AUG_RICH_DIR,cls)
    print(f'  {cls:<25}: {len(os.listdir(d)) if os.path.isdir(d) else 0}')

## Step 6 — Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from collections import Counter

class BadmintonDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        seq = np.load(path).astype(np.float32)   # (15, 102)
        T, F = seq.shape
        if T < FRAMES_PER_VIDEO:
            seq = np.vstack([seq, np.zeros((FRAMES_PER_VIDEO-T,F),np.float32)])
        elif T > FRAMES_PER_VIDEO:
            seq = seq[:FRAMES_PER_VIDEO]
        return torch.from_numpy(seq), torch.tensor(label, dtype=torch.long)

# Collect all augmented samples
all_samples = []
for li, cls in enumerate(ACTION_CLASSES):
    d = os.path.join(AUG_RICH_DIR, cls)
    if not os.path.isdir(d): continue
    for f in sorted(os.listdir(d)):
        if f.endswith('.npy'): all_samples.append((os.path.join(d,f), li))

# Hold-out test set: use only ORIGINAL files (no augmented ones)
orig_samples = []
for li, cls in enumerate(ACTION_CLASSES):
    d = os.path.join(RAW_KP_DIR, cls)
    if not os.path.isdir(d): continue
    for f in sorted(os.listdir(d)):
        if f.endswith('.npy'): orig_samples.append((os.path.join(d,f), li))

# Enrich originals on-the-fly for test
class RichDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        seq = np.load(path).astype(np.float32)
        T,F = seq.shape
        if T < FRAMES_PER_VIDEO: seq=np.vstack([seq,np.zeros((FRAMES_PER_VIDEO-T,F),np.float32)])
        seq = enrich(seq)   # (15,102)
        return torch.from_numpy(seq), torch.tensor(label, dtype=torch.long)

# Hold-out test (15% of originals)
orig_labels = [s[1] for s in orig_samples]
train_pool, test_orig = train_test_split(orig_samples, orig_labels,
    test_size=0.15, stratify=orig_labels, random_state=RANDOM_SEED)

test_loader = DataLoader(RichDataset(test_orig),
                         batch_size=64, shuffle=False, num_workers=2)

# The augmented samples for K-Fold (exclude any that correspond to test set)
test_stems = {os.path.splitext(os.path.basename(p))[0] for p,_ in test_orig}
train_aug = [(p,l) for p,l in all_samples
             if not any(stem in os.path.basename(p) for stem in test_stems)]

aug_labels = [s[1] for s in train_aug]

print(f'✅ Dataset ready')
print(f'   Augmented train pool : {len(train_aug):,}')
print(f'   Original test set    : {len(test_orig)}')
print(f'   K-Fold splits        : {N_FOLDS}')

## Step 7 — CNN-BiLSTM-Attention Model

In [ ]:
class CNNBiLSTMAttention(nn.Module):
    """
    Architecture:
      Input (B,15,102)
        ↓ LayerNorm
        ↓ 1D-CNN (local temporal patterns)
        ↓ BiLSTM (bidirectional sequence modeling)
        ↓ Multi-Head Attention (frame importance)
        ↓ Mean pooling
        ↓ FC → 6 classes
    """
    def __init__(self):
        super().__init__()
        self.input_norm = nn.LayerNorm(INPUT_SIZE)

        # 1D CNN — extracts local motion patterns over 3-frame windows
        self.cnn = nn.Sequential(
            nn.Conv1d(INPUT_SIZE, CNN_CHANNELS[0], kernel_size=3, padding=1),
            nn.BatchNorm1d(CNN_CHANNELS[0]), nn.GELU(), nn.Dropout(0.15),
            nn.Conv1d(CNN_CHANNELS[0], CNN_CHANNELS[1], kernel_size=3, padding=1),
            nn.BatchNorm1d(CNN_CHANNELS[1]), nn.GELU(), nn.Dropout(0.15),
        )
        # CNN output: (B, 256, T) → permute → (B, T, 256)

        # BiLSTM
        self.bilstm = nn.LSTM(
            CNN_CHANNELS[-1], HIDDEN_SIZE, NUM_LAYERS,
            batch_first=True, bidirectional=True,
            dropout=DROPOUT if NUM_LAYERS > 1 else 0.0)
        D = HIDDEN_SIZE * 2   # 512

        # Multi-Head Self-Attention
        self.mha      = nn.MultiheadAttention(D, MHA_HEADS, dropout=0.1, batch_first=True)
        self.mha_norm = nn.LayerNorm(D)

        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(D, FC_HIDDEN), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(FC_HIDDEN, NUM_CLASSES)
        )
        self._init()

    def _init(self):
        for n, p in self.bilstm.named_parameters():
            if 'weight_ih' in n:   nn.init.xavier_uniform_(p.data)
            elif 'weight_hh' in n: nn.init.orthogonal_(p.data)
            elif 'bias' in n:
                p.data.zero_()
                sz=p.size(0); p.data[sz//4:sz//2].fill_(1)

    def forward(self, x):
        x   = self.input_norm(x)          # (B,T,102)
        x   = x.permute(0,2,1)            # (B,102,T)
        x   = self.cnn(x)                 # (B,256,T)
        x   = x.permute(0,2,1)            # (B,T,256)
        lstm_out, _ = self.bilstm(x)      # (B,T,512)
        attn_out, _ = self.mha(lstm_out, lstm_out, lstm_out)  # (B,T,512)
        out = self.mha_norm(attn_out + lstm_out)  # residual
        ctx = out.mean(dim=1)             # (B,512) mean pooling
        return self.fc(ctx)               # (B,6)

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Smoke test
m = CNNBiLSTMAttention().to(device)
dummy = torch.randn(4, FRAMES_PER_VIDEO, INPUT_SIZE).to(device)
with torch.no_grad(): out = m(dummy)
print(f'✅ CNNBiLSTMAttention')
print(f'   Params : {m.n_params():,}')
print(f'   Input  : {list(dummy.shape)}  →  Output: {list(out.shape)}')

## Step 8 — 5-Fold Ensemble Training

In [ ]:
import copy, time
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
criterion   = nn.CrossEntropyLoss(label_smoothing=0.05)
fold_models = []
fold_accs   = []

def train_one_fold(fold_idx, tr_idx, vl_idx):
    print(f'\n{"="*55}')
    print(f'  FOLD {fold_idx+1}/{N_FOLDS}  '
          f'train={len(tr_idx):,}  val={len(vl_idx):,}')
    print(f'{"="*55}')

    tr_s = [train_aug[i] for i in tr_idx]
    vl_s = [train_aug[i] for i in vl_idx]

    tl = DataLoader(BadmintonDataset(tr_s), BATCH_SIZE, shuffle=True,
                    num_workers=2, pin_memory=True)
    vl = DataLoader(BadmintonDataset(vl_s), BATCH_SIZE, shuffle=False,
                    num_workers=2, pin_memory=True)

    mdl = CNNBiLSTMAttention().to(device)
    opt = AdamW(mdl.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sch = CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2, eta_min=1e-6)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)

    best_acc, patience, best_state = 0.0, 0, None

    for ep in range(1, NUM_EPOCHS+1):
        # ── Train ──────────────────────────────────────────────────
        mdl.train(); tl_c=tl_t=0
        for seqs, lbls in tl:
            seqs,lbls = seqs.to(device),lbls.to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(mdl(seqs), lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            opt.step()
            with torch.no_grad():
                tl_c+=(mdl(seqs).argmax(1)==lbls).sum().item(); tl_t+=lbls.size(0)
        sch.step()

        # ── Val ────────────────────────────────────────────────────
        mdl.eval(); vc=vt=0
        with torch.no_grad():
            for seqs, lbls in vl:
                seqs,lbls=seqs.to(device),lbls.to(device)
                vc+=(mdl(seqs).argmax(1)==lbls).sum().item(); vt+=lbls.size(0)
        v_acc = vc/vt

        if v_acc > best_acc:
            best_acc=v_acc; patience=0; best_state=copy.deepcopy(mdl.state_dict())
            print(f'  Ep {ep:>3} | Train {tl_c/tl_t:.1%} | Val {v_acc:.2%} ⭐')
        else:
            patience+=1
            if ep%25==0:
                print(f'  Ep {ep:>3} | Train {tl_c/tl_t:.1%} | Val {v_acc:.1%} | pat {patience}/{EARLY_STOP_PAT}')
        if patience >= EARLY_STOP_PAT:
            print(f'  Early stop ep {ep}'); break

    mdl.load_state_dict(best_state)
    CKPT = os.path.join(MODEL_DIR, f'fold{fold_idx+1}.pth')
    torch.save({'model_state_dict': mdl.state_dict(),
                'val_acc': best_acc, 'fold': fold_idx+1}, CKPT)
    print(f'  ✅ Fold {fold_idx+1} best val: {best_acc:.2%}  saved → {CKPT}')
    return mdl, best_acc


t0 = time.time()
for fi, (tr_idx, vl_idx) in enumerate(
        skf.split(train_aug, aug_labels)):
    fm, fa = train_one_fold(fi, tr_idx, vl_idx)
    fold_models.append(fm)
    fold_accs.append(fa)

print(f'\n{"="*55}')
print(f'  5-Fold Results:')
for i, a in enumerate(fold_accs):
    print(f'    Fold {i+1}: {a:.2%}')
print(f'  Mean : {np.mean(fold_accs):.2%}  ±  {np.std(fold_accs):.2%}')
print(f'  Time : {(time.time()-t0)/60:.1f} min')
print(f'{"="*55}')

## Step 9 — Ensemble Evaluation + TTA

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt, json
from sklearn.metrics import confusion_matrix, classification_report


def tta_predict(seqs_batch, models):
    """
    Test-Time Augmentation:
      Average predictions from each model over 2 views:
        1. Original sequence
        2. Horizontally flipped (x_new = 1 - x)
    Then average all model × view probabilities.
    """
    probs = torch.zeros(seqs_batch.size(0), NUM_CLASSES).to(device)
    seqs_flip = seqs_batch.clone()
    # Flip only position features (first 34 cols), alternate x columns
    seqs_flip[:, :, 0:34:2] = 1.0 - seqs_flip[:, :, 0:34:2]

    n = 0
    for m in models:
        m.eval()
        with torch.no_grad():
            probs += torch.softmax(m(seqs_batch), dim=1)  # original
            probs += torch.softmax(m(seqs_flip),  dim=1)  # flipped
        n += 2
    return (probs / n).cpu()


all_preds, all_labels, all_probs = [], [], []

for seqs, labels in test_loader:
    seqs = seqs.to(device)
    probs = tta_predict(seqs, fold_models)
    all_preds.extend(probs.argmax(1).tolist())
    all_labels.extend(labels.tolist())
    all_probs.extend(probs.tolist())

acc = sum(p==l for p,l in zip(all_preds,all_labels)) / len(all_labels)

print(f'\n{"="*55}')
print(f'  🏆  ENSEMBLE + TTA TEST ACCURACY: {acc:.2%}')
print(f'  📌  Paper LSTM baseline          : 80.00%')
print(f'  📌  Target                       : 94-95%')
print(f'  Model : CNN-BiLSTM-Attention ({fold_models[0].n_params():,} params × 5 folds)')
print(f'{"="*55}\n')
print(classification_report(all_labels, all_preds,
                             target_names=ACTION_CLASSES, digits=4))

# Per-fold accuracy on test set
print('Per-fold test accuracy:')
for i, m in enumerate(fold_models):
    m.eval(); c=t=0
    with torch.no_grad():
        for seqs, lbls in test_loader:
            out=m(seqs.to(device)); c+=(out.argmax(1)==lbls.to(device)).sum().item(); t+=lbls.size(0)
    print(f'  Fold {i+1}: {c/t:.2%}')
print(f'  Ensemble+TTA: {acc:.2%}')

# Save results
with open(f'{RESULTS_DIR}/results.json','w') as f:
    json.dump({'ensemble_acc':acc,'fold_val_accs':fold_accs,
               'predictions':all_preds,'labels':all_labels,
               'classes':ACTION_CLASSES}, f, indent=2)

In [ ]:
BG,PAN,TXT='#1A1A2E','#16213E','#E0E0E0'
cm   = confusion_matrix(all_labels, all_preds)
cm_n = cm.astype(float)/cm.sum(axis=1,keepdims=True)
short=[c.replace('_','\n') for c in ACTION_CLASSES]

fig,axes=plt.subplots(1,2,figsize=(18,7))
fig.suptitle(f'Ensemble + TTA  |  Test Accuracy: {acc:.2%}  |  5-Fold × {len(fold_models)} models',
             color=TXT,fontsize=14,fontweight='bold')
fig.patch.set_facecolor(BG)
for ax,data,fmt,title in[(axes[0],cm,'d','Raw'),(axes[1],cm_n,'.2f','Normalised')]:
    sns.heatmap(data,annot=True,fmt=fmt,cmap='RdPu',
                xticklabels=short,yticklabels=short,linewidths=0.5,linecolor=BG,ax=ax)
    ax.set_title(title,color=TXT);ax.set_xlabel('Predicted',color=TXT)
    ax.set_ylabel('Actual',color=TXT);ax.set_facecolor(PAN);ax.tick_params(colors=TXT,labelsize=8)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_ensemble.png',dpi=150,bbox_inches='tight')
plt.show()

# Per-class bar
pca=[cm.diagonal()[i]/cm.sum(axis=1)[i] for i in range(NUM_CLASSES)]
colors=['#6C63FF','#FF6584','#43D787','#FFD166','#F72585','#4CC9F0']
fig2,ax=plt.subplots(figsize=(10,5))
bars=ax.barh(ACTION_CLASSES,pca,color=colors,edgecolor=BG,height=0.55)
for b,v in zip(bars,pca): ax.text(v+0.01,b.get_y()+b.get_height()/2,f'{v:.1%}',va='center',color=TXT,fontsize=11)
ax.set_xlim(0,1.15);ax.set_xlabel('Accuracy',color=TXT)
ax.set_title('Per-Class Accuracy — 5-Fold Ensemble + TTA',color=TXT,fontsize=13,fontweight='bold')
ax.set_facecolor(PAN);fig2.patch.set_facecolor(BG);ax.tick_params(colors=TXT)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_class_ensemble.png',dpi=150,bbox_inches='tight')
plt.show()

print('\nPer-class accuracy:')
for cls,a in zip(ACTION_CLASSES,pca):
    print(f'  {cls:<25} {"█"*int(a*30):<30} {a:.1%}')

## Step 10 — Download Models

In [ ]:
from google.colab import files
import zipfile

# Zip all 5 fold checkpoints
zip_path = f'{MODEL_DIR}/ensemble_models.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for i in range(1, N_FOLDS+1):
        p = os.path.join(MODEL_DIR, f'fold{i}.pth')
        if os.path.exists(p): zf.write(p, f'fold{i}.pth')

print('📥 Downloading ensemble_models.zip ...')
files.download(zip_path)

print(f'\n── Final Summary ─────────────────────────────────────────')
print(f'  Dataset   : shenhuichang/badminton-storke-video (836 orig)')
print(f'  Augmented : ~{len(train_aug):,} samples')
print(f'  Features  : ({FRAMES_PER_VIDEO},{INPUT_SIZE}) pos+vel+acc')
print(f'  Model     : CNN-BiLSTM-Attention × {N_FOLDS} folds')
print(f'  Ensemble  : Avg softmax of {N_FOLDS}×2 predictions (TTA)')
print(f'  Test Acc  : {acc:.2%}')
print(f'  K-Fold Val: {np.mean(fold_accs):.2%} ± {np.std(fold_accs):.2%}')
print(f'──────────────────────────────────────────────────────────')

## Bonus — Predict Single Video

In [ ]:
from google.colab import files as cfiles
print('Upload a badminton video:')
uploaded = cfiles.upload()

for fname, data in uploaded.items():
    vp = f'/content/{fname}'
    open(vp,'wb').write(data)
    kp = extract_video(vp)
    if kp is None: print('❌ No pose detected'); continue
    rich_kp = torch.from_numpy(enrich(kp)).unsqueeze(0).to(device)  # (1,15,102)
    probs = tta_predict(rich_kp, fold_models).squeeze(0)
    pred = ACTION_CLASSES[probs.argmax().item()]
    print(f'\n🏸 {fname}  →  {pred}  ({probs.max():.1%} confidence)\n')
    for cls, p in zip(ACTION_CLASSES, probs):
        bar='█'*int(p.item()*30)
        tag='  ← predicted' if cls==pred else ''
        print(f'  {cls:<25} {bar:<30} {p.item():.1%}{tag}')